# TPGSR text-prior experiment

This is the next controlled experiment after Text Gestalt. It uses the official [TPGSR](https://github.com/mjq11302010044/TPGSR) `TSRN_TL` generator and its required pretrained CRNN. The CRNN reads the low-resolution image and produces character probabilities; those probabilities guide the super-resolution generator.

Important: the authors did not publish a ready generator checkpoint. Therefore this notebook trains a **single-stage reproduction**, not the complete published multi-stage model. It tests the central idea fairly on the same TextZoom hard subset and evaluates the result with PaddleOCR CER.

## 1. What this experiment changes

The earlier CNNs only received pixels. This model receives both the blurry pixels and a sequence of likely character probabilities from a frozen recogniser. Training uses pixel loss, stroke-gradient loss, and a gentle soft-prior loss that matches the recogniser's probability distributions on the restored and clear images. It does not force a hard CTC answer.

The official TextZoom test folders remain untouched during training.

In [1]:
from pathlib import Path
import os
import json
import subprocess
import sys
import pandas as pd
import torch
from IPython.display import Image, display

# Let unsupported MPS operations fall back to CPU instead of stopping the run.
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
print('PyTorch:', torch.__version__)
print('MPS built:', torch.backends.mps.is_built())
print('MPS available:', torch.backends.mps.is_available())
print('Device that training will use:',
      'cuda' if torch.cuda.is_available() else
      'mps' if torch.backends.mps.is_available() else 'cpu')

def find_repo(start=Path.cwd()):
    for folder in [start.resolve(), *start.resolve().parents]:
        if (folder / 'scripts' / 'train_tpgsr_textzoom.py').exists():
            return folder
    raise FileNotFoundError('Open this notebook from inside the whiteboard-ocr project')

REPO = find_repo()
SCRIPT = REPO / 'scripts' / 'train_tpgsr_textzoom.py'
OUTPUT = REPO / 'outputs' / 'tpgsr_textzoom'

# False runs a short pipeline check. True performs the meaningful experiment.
FULL_RUN = True

if FULL_RUN:
    # Resume the healthy epoch-2 pixel/stroke checkpoint and finish epochs 3-10.
    SETTINGS = ['--epochs', '10', '--batch-size', '16', '--val-limit', '512', '--resume']
else:
    SETTINGS = ['--epochs', '1', '--batch-size', '4', '--train-limit', '32',
                '--val-limit', '8', '--test-limit', '12', '--no-ocr']

print('Mode:', 'full experiment' if FULL_RUN else 'quick pipeline check')
print('Script:', SCRIPT)

PyTorch: 2.14.0
MPS built: True
MPS available: True
Device that training will use: mps
Mode: full experiment
Script: <LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/train_tpgsr_textzoom.py


## 2. Train and evaluate

The full run resumes the healthy epoch-2 warm-up checkpoint, trains epochs 3-10 with gradually introduced soft-prior matching, and saves a new checkpoint without overwriting the earlier CTC experiment. It then restores all 1,343 hard samples and runs PaddleOCR on bicubic, TPGSR and real HR images. This can take considerable time.

In [2]:
command = [sys.executable, str(SCRIPT), *SETTINGS]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

Running: <LOCAL_ASSIGNMENT>/.venv-paddle/bin/python <LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/train_tpgsr_textzoom.py --epochs 10 --batch-size 16 --val-limit 512 --resume
No Transformation module specified
No SequenceModeling module specified
resumed generator weights from epoch 2: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/tpgsr_single_stage_best.pt
device: mps; train=16,367; val=512
official generator: TSRN_TL; prior: frozen CRNN probabilities
loss schedule: 2 pixel/stroke warm-up epochs, then soft CRNN-prior matching ramps to weight 0.01
epoch 03: train=0.0847 val=0.0832 val_L1=0.0688 val_soft_prior=0.5071 train_prior_weight=0.0050 * best
epoch 04: train=0.0847 val=0.0838 val_L1=0.0694 val_soft_prior=0.5045 train_prior_weight=0.0100
epoch 05: train=0.0832 val=0.0839 val_L1=0.0696 val_soft_prior=0.5104 train_prior_weight=0.0100
epoch 06: train=0.0821 val=0.0824 val_L1=0.0684 val_soft_prior=0.4977 train_prior_weight=0.0100 * best
epoch 07: train=0.0812 val=0.0838 val_L1=

<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/skimage/metrics/simple_metrics.py:171: RuntimeWarning: divide by zero encountered in scalar divide
  return 10 * np.log10((data_range**2) / err)


  SR 200/1343
  SR 400/1343
  SR 600/1343
  SR 800/1343
  SR 1000/1343
  SR 1200/1343
  SR 1343/1343
OCR: bicubic


<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


  PaddleOCR 128/1343
  PaddleOCR 256/1343
  PaddleOCR 384/1343
  PaddleOCR 512/1343
  PaddleOCR 640/1343
  PaddleOCR 768/1343
  PaddleOCR 896/1343
  PaddleOCR 1024/1343
  PaddleOCR 1152/1343
  PaddleOCR 1280/1343
  PaddleOCR 1343/1343
OCR: tpgsr


Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


  PaddleOCR 128/1343
  PaddleOCR 256/1343
  PaddleOCR 384/1343
  PaddleOCR 512/1343
  PaddleOCR 640/1343
  PaddleOCR 768/1343
  PaddleOCR 896/1343
  PaddleOCR 1024/1343
  PaddleOCR 1152/1343
  PaddleOCR 1280/1343
  PaddleOCR 1343/1343
OCR: hr


Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


  PaddleOCR 128/1343
  PaddleOCR 256/1343
  PaddleOCR 384/1343
  PaddleOCR 512/1343
  PaddleOCR 640/1343
  PaddleOCR 768/1343
  PaddleOCR 896/1343
  PaddleOCR 1024/1343
  PaddleOCR 1152/1343
  PaddleOCR 1280/1343
  PaddleOCR 1343/1343
{
  "pairs": 1343,
  "bicubic_psnr": 20.508905027639134,
  "tpgsr_psnr": 20.47837304615985,
  "bicubic_ssim": 0.5863103257828821,
  "tpgsr_ssim": 0.617312549924535,
  "bicubic_cer": 0.4493305594779468,
  "tpgsr_cer": 0.4798994430931271,
  "bicubic_exact": 0.37751303052866714,
  "tpgsr_exact": 0.31347728965003724,
  "improved": 179,
  "unchanged": 845,
  "worsened": 319
}
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/tpgsr_results.csv
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/tpgsr_visual_comparison.png


CompletedProcess(args=['<LOCAL_ASSIGNMENT>/.venv-paddle/bin/python', '<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/train_tpgsr_textzoom.py', '--epochs', '10', '--batch-size', '16', '--val-limit', '512', '--resume'], returncode=0)

## 3. Read the result

The experiment succeeds only if TPGSR lowers PaddleOCR CER without causing more words to worsen than improve. HR is a reference showing how much recoverable information is missing from the LR input.

In [3]:
summary_path = OUTPUT / 'summary.json'
results_path = OUTPUT / 'tpgsr_results.csv'
figure_path = OUTPUT / 'tpgsr_visual_comparison.png'

summary = json.loads(summary_path.read_text())
display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))
display(Image(filename=str(figure_path)))
print('Per-image results:', results_path)

,value
pairs,1343.000000
bicubic_psnr,20.508905
tpgsr_psnr,20.478373
bicubic_ssim,0.586310
tpgsr_ssim,0.617313
bicubic_cer,0.449331
tpgsr_cer,0.479899
bicubic_exact,0.377513
tpgsr_exact,0.313477
improved,179.000000


<IPython.core.display.Image object>

Per-image results: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/tpgsr_results.csv


## 4. Interpretation rule

- Lower TPGSR CER than bicubic: text prior helped PaddleOCR overall.
- Higher exact-word accuracy: the improvement also produced more completely correct words.
- More improved than worsened samples: the method is reasonably consistent.
- Better PSNR but worse CER: it made the image visually closer without preserving useful character identity.

Even if it passes on TextZoom, handwritten whiteboard crops must still be evaluated separately because TextZoom mostly contains printed scene text.

## 5. Ablation: did the soft prior help?

This inference-only experiment compares bicubic enlargement, the saved epoch-2 pixel/stroke model, the saved epoch-9 soft-prior model, and real HR. It does not train or overwrite either checkpoint. Existing bicubic, soft-prior, and HR OCR results are reused, so only the missing epoch-2 output needs PaddleOCR evaluation.

In [4]:
ABLATION_SCRIPT = REPO / 'scripts' / 'compare_tpgsr_checkpoints.py'
command = [sys.executable, str(ABLATION_SCRIPT)]
print('Running inference-only comparison:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

Running inference-only comparison: <LOCAL_ASSIGNMENT>/.venv-paddle/bin/python <LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/compare_tpgsr_checkpoints.py
No Transformation module specified
No SequenceModeling module specified
device: mps; pixel/stroke checkpoint epoch: 2
Restoring 1,343 TextZoom hard images with epoch-2 model...
  SR 200/1343
  SR 400/1343
  SR 600/1343
  SR 800/1343
  SR 1000/1343
  SR 1200/1343
  SR 1343/1343
OCR: pixel/stroke checkpoint


<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


  PaddleOCR 128/1343
  PaddleOCR 256/1343
  PaddleOCR 384/1343
  PaddleOCR 512/1343
  PaddleOCR 640/1343
  PaddleOCR 768/1343
  PaddleOCR 896/1343
  PaddleOCR 1024/1343
  PaddleOCR 1152/1343
  PaddleOCR 1280/1343
  PaddleOCR 1343/1343
{
  "pairs": 1343,
  "bicubic_psnr": 20.508905027639134,
  "bicubic_ssim": 0.5863103257828821,
  "bicubic_cer": 0.4493305594779468,
  "bicubic_exact": 0.37751303052866714,
  "pixel_stroke_psnr": 20.04539790273475,
  "pixel_stroke_ssim": 0.5946546027472756,
  "pixel_stroke_cer": 0.48414742070790867,
  "pixel_stroke_exact": 0.3186895011169025,
  "pixel_stroke_improved": 175,
  "pixel_stroke_unchanged": 837,
  "pixel_stroke_worsened": 331,
  "soft_prior_psnr": 20.47837304615985,
  "soft_prior_ssim": 0.617312549924535,
  "soft_prior_cer": 0.4798994430931271,
  "soft_prior_exact": 0.31347728965003724,
  "soft_prior_improved": 179,
  "soft_prior_unchanged": 845,
  "soft_prior_worsened": 319,
  "hr_psnr": Infinity,
  "hr_ssim": 1.0,
  "hr_cer": 0.100659232593335

CompletedProcess(args=['<LOCAL_ASSIGNMENT>/.venv-paddle/bin/python', '<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/compare_tpgsr_checkpoints.py'], returncode=0)

## 6. Read the ablation result

In [5]:
ablation_summary_path = OUTPUT / 'checkpoint_ablation_summary.json'
ablation_results_path = OUTPUT / 'checkpoint_ablation_results.csv'
ablation_figure_path = OUTPUT / 'checkpoint_ablation_visual.png'

ablation_summary = json.loads(ablation_summary_path.read_text())
display(pd.DataFrame([ablation_summary]).T.rename(columns={0: 'value'}))
display(Image(filename=str(ablation_figure_path)))
print('Per-image ablation results:', ablation_results_path)

,value
pairs,1343.000000
bicubic_psnr,20.508905
bicubic_ssim,0.586310
bicubic_cer,0.449331
bicubic_exact,0.377513
pixel_stroke_psnr,20.045398
pixel_stroke_ssim,0.594655
pixel_stroke_cer,0.484147
pixel_stroke_exact,0.318690
pixel_stroke_improved,175.000000


<IPython.core.display.Image object>

Per-image ablation results: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/checkpoint_ablation_results.csv


## 7. Bicubic + TPGSR blend experiment

This experiment blends the safe bicubic image with the learned TPGSR image. Alpha 0 means all bicubic; alpha 1 means all TPGSR. The strength is selected using 488 held-out calibration pairs that were not used for training or checkpoint validation. Only the selected alpha is then evaluated on the untouched hard test set. No model is trained.

In [6]:
BLEND_SCRIPT = REPO / 'scripts' / 'evaluate_tpgsr_blend.py'
command = [sys.executable, str(BLEND_SCRIPT)]
print('Running blend calibration and test:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

Running blend calibration and test: <LOCAL_ASSIGNMENT>/.venv-paddle/bin/python <LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/evaluate_tpgsr_blend.py
No Transformation module specified
No SequenceModeling module specified


<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


device: mps; soft-prior checkpoint epoch: 9
Generating 488 untouched calibration pairs...
  SR 200/488
  SR 400/488
  SR 488/488
Calibration OCR: alpha=0.00
  PaddleOCR 128/488
  PaddleOCR 256/488
  PaddleOCR 384/488
  PaddleOCR 488/488
  CER=0.4353; exact=0.4139
Calibration OCR: alpha=0.05
  PaddleOCR 128/488
  PaddleOCR 256/488
  PaddleOCR 384/488
  PaddleOCR 488/488
  CER=0.4390; exact=0.4057
Calibration OCR: alpha=0.10
  PaddleOCR 128/488
  PaddleOCR 256/488
  PaddleOCR 384/488
  PaddleOCR 488/488
  CER=0.4360; exact=0.4139
Calibration OCR: alpha=0.20
  PaddleOCR 128/488
  PaddleOCR 256/488
  PaddleOCR 384/488
  PaddleOCR 488/488
  CER=0.4322; exact=0.4160
Calibration OCR: alpha=0.30
  PaddleOCR 128/488
  PaddleOCR 256/488
  PaddleOCR 384/488
  PaddleOCR 488/488
  CER=0.4230; exact=0.4119
Calibration OCR: alpha=0.50
  PaddleOCR 128/488
  PaddleOCR 256/488
  PaddleOCR 384/488
  PaddleOCR 488/488
  CER=0.4247; exact=0.3975
Calibration OCR: alpha=0.75
  PaddleOCR 128/488
  PaddleOCR 2

<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/evaluate_tpgsr_blend.py:221: UserWarning: Glyph 26085 (\N{CJK UNIFIED IDEOGRAPH-65E5}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/evaluate_tpgsr_blend.py:223: UserWarning: Glyph 26085 (\N{CJK UNIFIED IDEOGRAPH-65E5}) missing from font(s) DejaVu Sans.
  plt.savefig(visual_path, dpi=180, bbox_inches="tight")


saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/blend_calibration.csv
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/blend_calibration_curve.png
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/blend_test_results.csv
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/blend_summary.json
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/blend_visual_comparison.png


CompletedProcess(args=['<LOCAL_ASSIGNMENT>/.venv-paddle/bin/python', '<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/evaluate_tpgsr_blend.py'], returncode=0)

## 8. Read the blend result

In [7]:
blend_summary_path = OUTPUT / 'blend_summary.json'
blend_curve_path = OUTPUT / 'blend_calibration_curve.png'
blend_visual_path = OUTPUT / 'blend_visual_comparison.png'
blend_results_path = OUTPUT / 'blend_test_results.csv'

blend_summary = json.loads(blend_summary_path.read_text())
display(pd.DataFrame([blend_summary]).T.rename(columns={0: 'value'}))
display(Image(filename=str(blend_curve_path)))
display(Image(filename=str(blend_visual_path)))
print('Per-image blend results:', blend_results_path)

,value
selected_alpha,0.300000
calibration_cer,0.423004
calibration_exact,0.411885
test_pairs,1343.000000
bicubic_cer,0.449331
blend_cer,0.439146
tpgsr_cer,0.479899
bicubic_exact,0.377513
blend_exact,0.372301
tpgsr_exact,0.313477


<IPython.core.display.Image object>

<IPython.core.display.Image object>

Per-image blend results: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/blend_test_results.csv


## 9. Confidence-gated super-resolution

The fixed 30% blend helps some crops but harms others. This experiment runs PaddleOCR on both bicubic and blended crops, then uses the blend only when its recognition confidence is sufficiently higher. The confidence margin is selected on the separate calibration split while requiring calibration exact accuracy to remain at least as high as bicubic. The selected rule is then tested once on the untouched TextZoom hard test set.


In [8]:
GATE_SCRIPT = REPO / 'scripts' / 'evaluate_confidence_gated_sr.py'
command = [sys.executable, str(GATE_SCRIPT)]
print('Running confidence-gated comparison:', ' '.join(command))
subprocess.run(command, check=True)


Running confidence-gated comparison: <LOCAL_ASSIGNMENT>/.venv-paddle/bin/python <LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/evaluate_confidence_gated_sr.py
No Transformation module specified
No SequenceModeling module specified


<LOCAL_ASSIGNMENT>/.venv-paddle/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `<LOCAL_HOME>/.paddlex/official_models/PP-OCRv6_medium_rec`.


device: mps; checkpoint epoch: 9; fixed blend alpha: 0.30
Generating 488 calibration pairs...
  SR 200/488
  SR 400/488
  SR 488/488
Calibration OCR: bicubic
  PaddleOCR 128/488
  PaddleOCR 256/488
  PaddleOCR 384/488
  PaddleOCR 488/488
Calibration OCR: blend
  PaddleOCR 128/488
  PaddleOCR 256/488
  PaddleOCR 384/488
  PaddleOCR 488/488
margin=0.000: CER=0.4250; exact=0.4242; blend=247/488
margin=0.005: CER=0.4265; exact=0.4221; blend=179/488
margin=0.010: CER=0.4256; exact=0.4242; blend=164/488
margin=0.020: CER=0.4294; exact=0.4160; blend=143/488
margin=0.030: CER=0.4268; exact=0.4180; blend=129/488
margin=0.050: CER=0.4295; exact=0.4160; blend=108/488
margin=0.075: CER=0.4307; exact=0.4160; blend=89/488
margin=0.100: CER=0.4310; exact=0.4139; blend=78/488
margin=0.150: CER=0.4312; exact=0.4139; blend=61/488
margin=0.200: CER=0.4312; exact=0.4139; blend=47/488
margin=0.300: CER=0.4307; exact=0.4160; blend=22/488
margin=0.500: CER=0.4343; exact=0.4160; blend=2/488
margin=1.010: CER=

<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/evaluate_confidence_gated_sr.py:275: UserWarning: Glyph 26085 (\N{CJK UNIFIED IDEOGRAPH-65E5}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/evaluate_confidence_gated_sr.py:277: UserWarning: Glyph 26085 (\N{CJK UNIFIED IDEOGRAPH-65E5}) missing from font(s) DejaVu Sans.
  plt.savefig(visual_path, dpi=180, bbox_inches="tight")


saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/confidence_gate_calibration.csv
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/confidence_gate_calibration_curve.png
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/confidence_gate_test_results.csv
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/confidence_gate_summary.json
saved: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/confidence_gate_visual_comparison.png


CompletedProcess(args=['<LOCAL_ASSIGNMENT>/.venv-paddle/bin/python', '<LOCAL_ASSIGNMENT>/whiteboard-ocr/scripts/evaluate_confidence_gated_sr.py'], returncode=0)

## 10. Confidence-gate results

The main success condition is lower CER than the fixed blend while matching or exceeding bicubic exact-word accuracy. The selected-blend count also shows how often super-resolution was actually used.


In [9]:
gate_summary_path = OUTPUT / 'confidence_gate_summary.json'
gate_curve_path = OUTPUT / 'confidence_gate_calibration_curve.png'
gate_visual_path = OUTPUT / 'confidence_gate_visual_comparison.png'
gate_results_path = OUTPUT / 'confidence_gate_test_results.csv'

gate_summary = json.loads(gate_summary_path.read_text())
display(pd.DataFrame([gate_summary]).T.rename(columns={0: 'value'}))
display(Image(filename=str(gate_curve_path)))
display(Image(filename=str(gate_visual_path)))
print('Per-image confidence-gate results:', gate_results_path)


,value
blend_alpha,0.300000
selected_confidence_margin,0.000000
calibration_bicubic_cer,0.435316
calibration_bicubic_exact,0.413934
calibration_gated_cer,0.424952
calibration_gated_exact,0.424180
test_pairs,1343.000000
bicubic_cer,0.449331
blend_cer,0.439146
gated_cer,0.437034


<IPython.core.display.Image object>

<IPython.core.display.Image object>

Per-image confidence-gate results: <LOCAL_ASSIGNMENT>/whiteboard-ocr/outputs/tpgsr_textzoom/confidence_gate_test_results.csv
